# Aurora VIII · worker de entrenamiento en Google Colab

Alternativa gratuita al *Background Worker* de Render. Este cuaderno toma trabajos de la cola en Supabase, entrena con TensorFlow/Keras y publica el progreso; la interfaz en Vercel lo muestra en vivo.

Colab corta la sesión tras un tiempo de inactividad: déjalo abierto mientras haya entrenamientos. Si se corta a mitad de un trabajo, la plataforma lo reencola sola.

In [ ]:
REPO = "https://github.com/TU_USUARIO/aurora-viii.git"  # tu repositorio
!git clone -q $REPO aurora-viii || (cd aurora-viii && git pull -q)
!pip install -q -r aurora-viii/services/api/requirements.txt

TensorFlow ya viene instalado en Colab. Ingresa los secretos (no se guardan en el cuaderno).

In [ ]:
import os, getpass
os.environ['DATABASE_URL'] = getpass.getpass('DATABASE_URL (Session pooler de Supabase): ')
os.environ['SUPABASE_URL'] = input('SUPABASE_URL (https://TU_REF.supabase.co): ')
os.environ['SUPABASE_SECRET_KEY'] = getpass.getpass('SUPABASE_SECRET_KEY: ')
os.environ.update(STORAGE_BACKEND='supabase', STORAGE_BUCKET='aurora-viii', ENVIRONMENT='prod',
                  MIGRATIONS_DIR='/content/aurora-viii/infra/migrations')

In [ ]:
import sys
sys.path[:0] = ['/content/aurora-viii/services/api', '/content/aurora-viii/workers/training']
from app.core.logging import setup_logging
from app.core.migrate import run_migrations
setup_logging('INFO'); run_migrations()
from aurora_worker.runner import run_loop
run_loop(worker_id='colab')  # detén la celda para parar el worker